<a href="https://colab.research.google.com/github/gustkt/GE338-Lab-3/blob/main/lab3_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lab 3: Machine Learning สำหรับการจำแนกการใช้ที่ดิน**

**Situation**

องค์กรอนุรักษ์ต้องการแผนที่การใช้ที่ดินปัจจุบันของพื้นที่ชุ่มน้ำและพื้นที่เกษตรกรรมที่อยู่ติดกันในภาคกลางหรือภาคตะวันออกของไทย พวกเขามีงบจำกัดและไม่สามารถลงพื้นที่เก็บ Ground Truth ได้ครบทุกจุด ต้องการรู้ว่า

• วิธีไหนให้ผลแม่นยำที่สุดเมื่อ Training Data มีน้อย?

• ถ้าใช้แค่ Spectral Bands ปกติ vs เพิ่ม Spectral Indices เข้าไป — ต่างกันแค่ไหน?

• แผนที่ที่ได้เชื่อถือได้แค่ไหน และในบริเวณใดบ้างที่ไม่แน่ใจ?


* **ภารกิจที่ 1: ออกแบบ Training Strategy**

▸ จะใช้ประเภทที่ดิน 5 class ได้แก่
Water (แหล่งน้ำ), Agriculture (เกษตรกรรม) ,Forest (ป่าไม้), Built-up (สิ่งปลูกสร้าง) และ Bareland (พื้นดินโล่ง)

▸ จะหา Training Samples โดยการวาด Polygon เองผ่าน Google Earth Engine Code Editor

▸ จะแบ่ง Train/Validation อย่างไร? — 80/20 แบบสุ่ม vs Spatial Cross-validation ต่างกันอย่างไร?
80/20

▸ Feature ที่จะใช้มีอะไรบ้าง? —
Red, Green, Blue, NIR, SWIR, NDVI, NDWI, SAVI เพราะว่า Red, Green, Blue เป็น feature พื้นฐาน
NIR, SWIR ใช้แยกความชื้น, น้ำ
NDVI ใช้แยก vegetation ได้ดี
NDWI ใช้แยกแหล่งน้ำได้ดี
SAVI ใช้แยก vegetation ที่ปนกับ Bareland ได้ดีขึ้น


In [ ]:
# ติดตั้ง + import
!pip install geemap xgboost pycrs -q

In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, cohen_kappa_score
from sklearn.metrics import classification_report

In [ ]:
# Authenticate + Initialize
ee.Authenticate()
ee.Initialize(project='ee-gustkt45513')

In [ ]:
# ดึงขอบเขตจังหวัดตาก จาก GEE

# ใช้ FAO GAUL dataset
roiPT = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM1_NAME', 'Tak'))

# ฟังก์ชัน mask cloud (SCL)
def mask_s2_clouds_scl(image):
    scl = image.select('SCL')

# คลาสที่เราต้องการ "กรองออก" (Mask out)
  # 3: Cloud Shadow
  # 8: Cloud Medium Probability
  # 9: Cloud High Probability
  # 10: Thin Cirrus
  # 11: Snow
    bad_classes = [3, 8, 9, 10]

    mask = scl.remap(
        bad_classes,
        ee.List.repeat(0, len(bad_classes)),
        1
    )

    return image.updateMask(mask).divide(10000)

# โหลด Sentinel-2
dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2026-01-01', '2026-01-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .filterBounds(roiPT)
    .map(mask_s2_clouds_scl)
    .select(['B2','B3','B4','B8','B11'])
)

image = dataset.median().clip(roiPT)

# แสดง Map
Map = geemap.Map()

# แสดงภาพ RGB
Map.addLayer(image, {
    'min': 0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2']
}, 'Sentinel-2 RGB')

# แสดงขอบเขต ROI
Map.addLayer(roiPT, {}, 'ROI')

# zoom ไปที่พื้นที่
Map.centerObject(roiPT, 8)

Map

In [ ]:
# สร้าง Features (Bands + Indices)
# NDVI
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDWI
ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

# SAVI
savi = image.expression(
    '((NIR - RED) / (NIR + RED + L)) * (1 + L)', {
        'NIR': image.select('B8'),
        'RED': image.select('B4'),
        'L': 0.5
}).rename('SAVI')

# stack
stack = image.select(['B2','B3','B4','B8','B11']).addBands([ndvi, ndwi, savi])

In [ ]:
# โหลด Training Samples (จากที่ทำใน GEE Code Editer)
training_fc = geemap.shp_to_ee("/content/drive/MyDrive/Data_for_data_science/training_sample_50/Training_samples_all.shp")

# เช็ค Attribute Table ดูข้อมูลของ Training samples ที่ทำมา
# แปลง FeatureCollection → pandas
df_samples = geemap.ee_to_df(training_fc)

# ดูข้อมูล
print("Shape:", df_samples.shape)
display(df_samples.head())

mapping = df_samples[['Class', 'Class_EN']].drop_duplicates().sort_values('Class')
display(mapping)

print("\nColumns:")
print(df_samples.columns)

In [ ]:
# Sample pixel values
samples = stack.sampleRegions(
    collection=training_fc,
    properties=['Class'],
    scale=10
)

# แปลงเป็น Pandas
df = geemap.ee_to_df(samples)
df = df.dropna()

X = df.drop(columns=['Class'])
y = df['Class']

In [ ]:
# Train/Test Split แบบ Random 80/20

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

* **ภารกิจที่ 2: เปรียบเทียบอัลกอริทึม**

In [ ]:
# Case 1 : Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [ ]:
# Case 2 : XGBoost
y_train_xgb = y_train
y_test_xgb = y_test

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=len(y_train_xgb.unique())
)
xgb.fit(X_train, y_train_xgb)

xgb_pred = xgb.predict(X_test)


In [ ]:
# Confusion Matrix
cm_rf = confusion_matrix(y_test, rf_pred)
cm_xgb = confusion_matrix(y_test, xgb_pred)

print("Confusion Matrix RF:\n", cm_rf)
print("Confusion Matrix XGB:\n", cm_xgb)

In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d')
plt.title("Confusion Matrix (RF)")
# plt.savefig('Confusion Matrix (Random Forest).png', dpi=300) # เซฟรูปลงโฟลเดอร์ sample_data ในเมนูไฟล์ด้านซ้าย
plt.show()
# Class 0 = Water (แหล่งน้ำ)
# Class 1 = Agriculture (เกษตรกรรม)
# Class 2 = Forest (ป่าไม้)
# Class 3 = Built-up (สิ่งปลูกสร้าง)
# Class 4 = Bareland (พื้นดินโล่ง)

เมื่อพิจารณาเฉพาะความผิดพลาดในการจำแนกจาก confusion matrix ของ RF พบว่ามีความผิดพลาดในการจำแนกจาก Bareland (พื้นดินโล่ง) ไปเป็น Built-up (สิ่งปลูกสร้าง) มากที่สุดเนื่องจาก Landuse ทั้ง 2 ชนิดมักมีค่าการสะท้อนที่ใกล้เคียงกัน

In [ ]:
cm_xgb = confusion_matrix(y_test, xgb_pred)
sns.heatmap(cm_xgb, annot=True, fmt='d')
plt.title("Confusion Matrix (XGB)")
# plt.savefig('Confusion Matrix (XGBoost).png', dpi=300) # เซฟรูปลงโฟลเดอร์ sample_data ในเมนูไฟล์ด้านซ้าย
plt.show()
# Class 0 = Water (แหล่งน้ำ)
# Class 1 = Agriculture (เกษตรกรรม)
# Class 2 = Forest (ป่าไม้)
# Class 3 = Built-up (สิ่งปลูกสร้าง)
# Class 4 = Bareland (พื้นดินโล่ง)

เมื่อพิจารณาเฉพาะความผิดพลาดในการจำแนกจาก confusion matrix ของ XGB พบว่ามีความผิดพลาดในการจำแนกนอกจาก Bareland (พื้นดินโล่ง) ไปเป็น Built-up (สิ่งปลูกสร้าง) ด้วยเหตุผลที่กล่าวไปเช่นเดียวกับโมเดล RF แล้วยังพบการจำแนก Water (แหล่งน้ำ) ไปเป็น Agriculture (เกษตรกรรม) เนื่องจากบริบทของการทำนาข้าวที่บางพื้นที่รอบล้อมไปด้วยน้ำทำให้ได้ค่าการสะท้อนที่เป็นค่าของน้ำจึงทำให้โมเดลเกิด confuse ได้

In [ ]:
# Producer’s / User’s Accuracy

def accuracy_metrics(cm):
    # Producer's Accuracy = Recall (row-based)
    producer_acc = np.diag(cm) / np.sum(cm, axis=1)

    # User's Accuracy = Precision (column-based)
    user_acc = np.diag(cm) / np.sum(cm, axis=0)

    return producer_acc, user_acc

pa_rf, ua_rf = accuracy_metrics(cm_rf)
pa_xgb, ua_xgb = accuracy_metrics(cm_xgb)

print("RF Producer's Accuracy:", pa_rf)
print("RF User's Accuracy:", ua_rf)

print("XGB Producer's Accuracy:", pa_xgb)
print("XGB User's Accuracy:", ua_xgb)

In [ ]:
# F1-score ต่อ Class
f1_rf = f1_score(y_test, rf_pred, average=None)
f1_xgb = f1_score(y_test, xgb_pred, average=None)

print("F1-score RF:", f1_rf)
print("F1-score XGB:", f1_xgb)

In [ ]:
# Kappa
print("Kappa RF:", cohen_kappa_score(y_test, rf_pred))
print("Kappa XGB:", cohen_kappa_score(y_test, xgb_pred))

In [ ]:
# ทำเป็น Dataframe
classes = np.unique(y_test)

df_metrics = pd.DataFrame({
    'Class': classes,
    'Producer_Accuracy_RF': pa_rf,
    'User_Accuracy_RF': ua_rf,
    'F1_RF': f1_rf,
    'Producer_Accuracy_XGB': pa_xgb,
    'User_Accuracy_XGB': ua_xgb,
    'F1_XGB': f1_xgb
})

# เอา mapping Class → Class_EN
class_mapping = df_samples[['Class', 'Class_EN']].drop_duplicates()
class_mapping = class_mapping.sort_values('Class').reset_index(drop=True)

# merge เข้ากับ df_metrics
df_metrics = df_metrics.merge(class_mapping, on='Class', how='left')

df_metrics = df_metrics[
    ['Class', 'Class_EN',
     'Producer_Accuracy_RF', 'User_Accuracy_RF', 'F1_RF',
     'Producer_Accuracy_XGB', 'User_Accuracy_XGB', 'F1_XGB']
]
print(df_metrics)

โดยภาพรวม RF และ XGB สามารถจำแนกการใช้ประโยชน์ที่ดินได้ดี แต่ XGB ให้ performance ที่เสถียรกว่าในหลาย class

Class Water RF มี PA,UA อยู่ที่ 0.92 ส่วน XGB PA,UA อยู่ที่ 1.00 และ 0.87 ตามลำดับ แสดงให้เห็นว่า XGB จับ Water ได้หมด แต่มี false positive ที่โมเดลสับสนระหว่าง water กับ agriculture

Class Agriculture ทั้ง 2 โมเดลมีค่า PA, UA อยู่ที่ 0.57 และ 1.00 ตามลำดับ แสดงให้เห็นถึงความ conservative ของโมเดลที่มักทำนายโดยใช้ค่าค่าเฉลี่ย (Mean/Median)

Class Forest RF มี PA,UA อยู่ที่ 1.00 และ 0.83 ส่วน XGB PA,UA อยู่ที่ 1.00 แสดงให้เห็นว่า Forest มี spectral signature ที่ชัดเจนทำให้ทั้ง 2 โมเดลทำนายได้ดีเยี่ยม โดยเฉพาะ XGB ที่ทำนายถูกต้องทั้งหมด

Class Buit-up RF มี PA อยู่ที่ 0.73 ส่วน XGB PA อยู่ที่ 0.82 แสดงให้เห็นว่า XGB ทำนายได้ดีกว่าแต่ก็ยังมี confusion อยู่

Class Bareland RF มี UA อยู่ที่ 0.69 ส่วน XGB UA อยู่ที่ 0.75 แสดงให้เห็นว่ามี false positive เยอะ มีความสับสนกับ Agriculture และ Built-up

* **ภารกิจที่ 3: วิเคราะห์ Feature Importance**

In [ ]:
# ดึง Feature Importance (RF)
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance)

# plot
importance.plot(kind='bar')
plt.title("Feature Importance (Random Forest)")
# plt.savefig('Feature Importance (Random Forest).png', dpi=300) # เซฟรูปลงโฟลเดอร์ sample_data ในเมนูไฟล์ด้านซ้าย
plt.show()

จากการที่ดึง Feature Importance ของ RF ออกมาพบว่า NDVI มีความสำคัญมากที่สุดโดยมีค่าอยู่ที่ 0.179783 ซึ่งสอดคล้องกับลักษณะทางกายภาพของพื้นที่ที่ส่วนใหญ่เป็นพื้นที่พืชพรรณ เช่น ป่าไม้ หรือเกษตรกรรม ในขณะที่ B4 (Red) มีความสำคัญน้อยที่สุดมีค่าอยู่ที่ 0.064336 ซึ่งอาจเป็นผลมาจาก Band Red เป็น Band ที่ใช้ในการคำนวณดัชนีทั้ง NDVI และ SAVI จึงเกิดความซ้ำซ้อนของ feature ทำให้ Band Red เป็น feature ที่มีความสำคัญน้อยที่สุด

In [ ]:
# เลือก top 3 features
top_features = importance.head(3).index

rf_reduced = RandomForestClassifier(n_estimators=100, random_state=42)
rf_reduced.fit(X_train[top_features], y_train)

pred_reduced = rf_reduced.predict(X_test[top_features])

print(classification_report(y_test, pred_reduced))

In [ ]:
# confusion matrix
cm_rf_reduced = confusion_matrix(y_test, pred_reduced)

# function เดิม
def accuracy_metrics(cm):
    producer_acc = np.diag(cm) / np.sum(cm, axis=1)
    user_acc = np.diag(cm) / np.sum(cm, axis=0)
    return producer_acc, user_acc

pa_rf_reduced, ua_rf_reduced = accuracy_metrics(cm_rf_reduced)

# F1 per class
f1_rf_reduced = f1_score(y_test, pred_reduced, average=None)

In [ ]:
classes = np.unique(y_test)

df_metrics = pd.DataFrame({
    'Class': classes,

    # RF (เดิม)
    'PA_RF': pa_rf,
    'UA_RF': ua_rf,
    'F1_RF': f1_rf,

    # XGB
    'PA_XGB': pa_xgb,
    'UA_XGB': ua_xgb,
    'F1_XGB': f1_xgb,

    # 🔥 RF Reduced (ใหม่)
    'PA_RF_reduced': pa_rf_reduced,
    'UA_RF_reduced': ua_rf_reduced,
    'F1_RF_reduced': f1_rf_reduced
})

In [ ]:
cm_rf_reduced = confusion_matrix(y_test, pred_reduced)
sns.heatmap(cm_rf_reduced, annot=True, fmt='d')
plt.title("Confusion Matrix (RF_reduced)")
# plt.savefig('Confusion Matrix (RF_reduced).png', dpi=300) # เซฟรูปลงโฟลเดอร์ sample_data ในเมนูไฟล์ด้านซ้าย
plt.show()
# Class 0 = Water (แหล่งน้ำ)
# Class 1 = Agriculture (เกษตรกรรม)
# Class 2 = Forest (ป่าไม้)
# Class 3 = Built-up (สิ่งปลูกสร้าง)
# Class 4 = Bareland (พื้นดินโล่ง)

In [ ]:
# mapping
class_mapping = df_samples[['Class', 'Class_EN']].drop_duplicates()
class_mapping = class_mapping.sort_values('Class').reset_index(drop=True)

# merge
df_metrics = df_metrics.merge(class_mapping, on='Class', how='left')

# จัด column
df_metrics = df_metrics[
    ['Class', 'Class_EN',
     'PA_RF', 'UA_RF', 'F1_RF',
     'PA_XGB', 'UA_XGB', 'F1_XGB',
     'PA_RF_reduced', 'UA_RF_reduced', 'F1_RF_reduced']
]

print(df_metrics)

In [ ]:
# ภาพรวมของทั้ง 3 โมเดล
df_summary = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'RF (Reduced Features)'],

    # Overall Accuracy
    'Overall_Accuracy': [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y_test, pred_reduced)
    ],

    # Kappa
    'Kappa': [
        cohen_kappa_score(y_test, rf_pred),
        cohen_kappa_score(y_test, xgb_pred),
        cohen_kappa_score(y_test, pred_reduced)
    ],

    # F1 (macro)
    'F1_macro': [
        f1_score(y_test, rf_pred, average='macro'),
        f1_score(y_test, xgb_pred, average='macro'),
        f1_score(y_test, pred_reduced, average='macro')
    ],

    # Precision (macro)
    'Precision_macro': [
        precision_score(y_test, rf_pred, average='macro'),
        precision_score(y_test, xgb_pred, average='macro'),
        precision_score(y_test, pred_reduced, average='macro')
    ],

    # Recall (macro)
    'Recall_macro': [
        recall_score(y_test, rf_pred, average='macro'),
        recall_score(y_test, xgb_pred, average='macro'),
        recall_score(y_test, pred_reduced, average='macro')
    ]
})

print(df_summary)

การตัด Feature ที่มีความสำคัญต่ำออก เลือก Feature ที่มีความสำคัญสูงสุด 3 อันดับแรกทำให้โมเดล RF มี Overall Accuracy สูงขึ้นเล็กน้อยจาก 0.86 เป็น 0.88

* **ภารกิจที่ 4: ประเมินความไม่แน่นอน**

แผนที่การจำแนกทุกแผนที่มีบริเวณที่ไม่แน่นอน นักศึกษาต้องระบุและแสดงให้ได้ว่า

▸ Class ใดถูกสับสนกันมากที่สุดใน Confusion Matrix? เพราะอะไรในทางกายภาพ?

▸ บริเวณที่โมเดลไม่แน่ใจ (Class Probability ต่ำหรือใกล้เคียงกัน) อยู่ที่ไหน?

▸ ถ้าต้องบอก Stakeholder ว่าแผนที่นี้เชื่อถือได้แค่ไหน จะพูดอย่างไร?


In [ ]:
# คำนวณ Probability
probs = rf.predict_proba(X_test)

# confidence ของแต่ละ sample
confidence = np.max(probs, axis=1)

print("Mean confidence:", confidence.mean())

จากค่า Mean confidence ที่หากค่าเข้าใกล้ 1 แสดงถึงโมเดลมีความมั่นใจในการทำนาย ซึ่งโมเดล RF ในครั้งนี้อยู่ที่ประมาณ 0.86 เป็นค่าที่ค่อนข้างสูงซึ่งชี้ให้เห็นว่าโมเดลตัวนี้เชื่อถือได้ในภาพรวม

In [ ]:
# หา “จุดที่ไม่แน่ใจ”
uncertain_idx = confidence < 0.6
uncertain_samples = X_test[uncertain_idx]

print("Number of uncertain samples:", len(uncertain_samples))

uncertain_classes = y_test[uncertain_idx]
print(uncertain_classes.value_counts())
# Class 0 = Water (แหล่งน้ำ)
# Class 1 = Agriculture (เกษตรกรรม)
# Class 2 = Forest (ป่าไม้)
# Class 3 = Built-up (สิ่งปลูกสร้าง)
# Class 4 = Bareland (พื้นดินโล่ง)

In [ ]:
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix (RF)")
plt.show()
# Class 0 = Water (แหล่งน้ำ)
# Class 1 = Agriculture (เกษตรกรรม)
# Class 2 = Forest (ป่าไม้)
# Class 3 = Built-up (สิ่งปลูกสร้าง)
# Class 4 = Bareland (พื้นดินโล่ง)

เมื่อพิจารณาเฉพาะความผิดพลาดในการจำแนกจาก confusion matrix ของ RF พบว่ามีความผิดพลาดในการจำแนกจาก Bareland (พื้นดินโล่ง) ไปเป็น Built-up (สิ่งปลูกสร้าง) มากที่สุดเนื่องจาก Landuse ทั้ง 2 ชนิด ตัวอย่างเช่น หลังคา กับพื้นดินโล่ง ๆ   มีค่าการสะท้อนที่ใกล้เคียงกันทำให้โมเดลเกิดการ confuse ได้

* คำถามสำหรับใช้คิดและตอบใน README









1. ถ้าเพิ่ม Training Samples อีก 2 เท่า ความแม่นยำจะเพิ่มขึ้นเท่าไหร่? ทดสอบและอธิบายผล

In [ ]:
# โหลด Training Samples 2X (จากที่ทำใน GEE Code Editer)
df_2x = geemap.shp_to_ee("/content/drive/MyDrive/Data_for_data_science/training_sample_100/Training_samples_100.shp")

# เช็ค Attribute Table ดูข้อมูลของ Training samples ที่ทำมา
# แปลง FeatureCollection → pandas
df_samples = geemap.ee_to_df(df_2x)

# ดูข้อมูล
print("Shape:", df_samples.shape)
display(df_samples.head())

mapping = df_samples[['Class', 'Class_EN']].drop_duplicates().sort_values('Class')
display(mapping)

print("\nColumns:")
print(df_samples.columns)

In [ ]:
# Sample pixel values
samples = stack.sampleRegions(
    collection=df_2x,
    properties=['Class'],
    scale=10
)

# แปลงเป็น Pandas
df = geemap.ee_to_df(samples)
df = df.dropna()

X = df.drop(columns=['Class'])
y = df['Class']

In [ ]:
# Train/Test Split แบบ Random 80/20

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Case 1 : Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

# Case 2 : XGBoost
y_train_xgb = y_train
y_test_xgb = y_test

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=len(y_train_xgb.unique())
)
xgb.fit(X_train, y_train_xgb)

xgb_pred = xgb.predict(X_test)


In [ ]:
# Confusion Matrix
cm_rf = confusion_matrix(y_test, rf_pred)
cm_xgb = confusion_matrix(y_test, xgb_pred)

print("Confusion Matrix RF:\n", cm_rf)
print("Confusion Matrix XGB:\n", cm_xgb)

In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d')
plt.title("Confusion Matrix (RF_sample_X2)")
plt.show()

# Class 0 = Water (แหล่งน้ำ)
# Class 1 = Agriculture (เกษตรกรรม)
# Class 2 = Forest (ป่าไม้)
# Class 3 = Built-up (สิ่งปลูกสร้าง)
# Class 4 = Bareland (พื้นดินโล่ง)

cm_xgb = confusion_matrix(y_test, xgb_pred)
sns.heatmap(cm_xgb, annot=True, fmt='d')
plt.title("Confusion Matrix (XGB_sample_X2)")
plt.show()

In [ ]:
# Producer’s / User’s Accuracy

def accuracy_metrics(cm):
    # Producer's Accuracy = Recall (row-based)
    producer_acc = np.diag(cm) / np.sum(cm, axis=1)

    # User's Accuracy = Precision (column-based)
    user_acc = np.diag(cm) / np.sum(cm, axis=0)

    return producer_acc, user_acc

pa_rf, ua_rf = accuracy_metrics(cm_rf)
pa_xgb, ua_xgb = accuracy_metrics(cm_xgb)

# F1-score ต่อ Class
f1_rf = f1_score(y_test, rf_pred, average=None)
f1_xgb = f1_score(y_test, xgb_pred, average=None)

# Kappa
print("Kappa RF:", cohen_kappa_score(y_test, rf_pred))
print("Kappa XGB:", cohen_kappa_score(y_test, xgb_pred))

In [ ]:
# ทำเป็น Dataframe
classes = np.unique(y_test)

df_metrics = pd.DataFrame({
    'Class': classes,
    'Producer_Accuracy_RF': pa_rf,
    'User_Accuracy_RF': ua_rf,
    'F1_RF': f1_rf,
    'Producer_Accuracy_XGB': pa_xgb,
    'User_Accuracy_XGB': ua_xgb,
    'F1_XGB': f1_xgb
})

# เอา mapping Class → Class_EN
class_mapping = df_samples[['Class', 'Class_EN']].drop_duplicates()
class_mapping = class_mapping.sort_values('Class').reset_index(drop=True)

# merge เข้ากับ df_metrics
df_metrics = df_metrics.merge(class_mapping, on='Class', how='left')

df_metrics = df_metrics[
    ['Class', 'Class_EN',
     'Producer_Accuracy_RF', 'User_Accuracy_RF', 'F1_RF',
     'Producer_Accuracy_XGB', 'User_Accuracy_XGB', 'F1_XGB']
]
print(df_metrics)

In [ ]:
# ภาพรวมของทั้ง 3 โมเดล
df_summary = pd.DataFrame({
    'Model': ['Random Forest_V2', 'XGBoost_V2'],

    # Overall Accuracy
    'Overall_Accuracy': [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred),
    ]
    })

print(df_summary)

เมื่อเพิ่ม Training Samples อีก 2 เท่า ความแม่นยำในเชิง Overall Accuracy RF จากเดิม 0.86 เพิ่มขึ้นเป็น 0.90 ในขณะที่ XGBoost มี OA ลดลงเล็กน้อยจากเดิม 0.90 เป็น 0.89

2. Spatial Autocorrelation ในข้อมูล Training มีผลต่อความแม่นยำที่รายงานอย่างไร?

In [ ]:
# Sample pixel values
samples = stack.sampleRegions(
    collection=training_fc,
    properties=['Class', 'POINT_X', 'POINT_Y'], # Include POINT_X and POINT_Y
    scale=10
)

# แปลงเป็น Pandas
df = geemap.ee_to_df(samples)
df = df.dropna()

X = df.drop(columns=['Class', 'POINT_X', 'POINT_Y']) # Drop POINT_X, POINT_Y from features for training
y = df['Class']

In [ ]:
# ตัวอย่าง spatial split แบบง่าย: แบ่ง ROI เป็นซ้าย/ขวา
df['fold'] = (df['POINT_X'] > df['POINT_X'].median()).astype(int)

train_df = df[df['fold'] == 0]
test_df  = df[df['fold'] == 1]

# train/test
X_tr, y_tr = train_df.drop(columns=['Class','fold']), train_df['Class']
X_te, y_te = test_df.drop(columns=['Class','fold']), test_df['Class']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_te)

print("Spatial OA:", accuracy_score(y_te, pred))

จากการทดลอง Run โค้ดเพิ่มเติม Spatial Autocorrelation ในข้อมูล Training มีผลต่อความแม่นยำสูงกว่า แบบแบ่ง 80/20 แล้วสุ่มเล็กน้อย (0.86 กับ 0.88)

3. Class ใดที่โมเดลทำได้แย่ที่สุด — แก้ได้ด้วยวิธีใดบ้าง?

Class Agriculture โมเดล RF ทำได้แย่สุด ถึงแม้จะมีค่า UA = 1.00 แต่ PA,F1-score มีค่า 0.57 และ 0.72 ตามลำดับ ซึ่งอาจแก้ได้ด้วยการเพิ่ม training samples, ใช้ NDVI time-series ที่ช่วยให้สามารถแยกนาข้าวได้ดีขึ้น หรือจะแก้ด้วยการปรับ class definition แยกเป็นนาข้าวกับเกษตรกรรมอื่น ๆ หรือพื้นที่เกษตรแบบเปียกกับพื้นที่เกษตรแบบแห้ง

4. ถ้าต้องทำซ้ำ Lab นี้สำหรับพื้นที่อื่น อะไรคือสิ่งที่ต้องเปลี่ยน และอะไรที่ใช้ซ้ำได้?

ถ้าจะเปลี่ยนพื้นที่ศึกษาสิ่งที่ใช้ซ้ำได้ คือ ขั้นตอนการรันโค้ด (Code pipeline) ส่วน Class definition,Feature ที่ใช้, Model, Evalution metrics อาจจะใช้ซ้ำหรือไม่ใช้ก็ได้ขึ้นอยู่กับบริบทของพื้นที่ แต่สิ่งที่ต้องเปลี่ยนคือ training samples ต้องทำการสุ่มใหม่ทุกพื้นที่